# Task 3 — Dual Local Models Blind A/B for Kaggle (`Save Version`) + `gdown`

Этот вариант notebook рассчитан на сценарий **Kaggle Save Version / Run All**:

- весь orchestration вынесен в `.py`-скрипт;
- notebook сам подготавливает окружение и запускает CLI;
- ZIP с экспертным YAML скачивается через `!gdown` с проверкой, что реально получен ZIP, а не HTML-страница Google Drive;
- репозиторий ставится из официального GitHub-URL `https://github.com/top-papers/top-papers-graph` и при необходимости может быть переиспользован из уже существующей локальной папки;
- `git clone` вызывается **только если** локальный репозиторий не найден и хост GitHub реально резолвится, чтобы не падать на ошибке `Could not resolve host: github.com`;
- все выходные файлы сохраняются в `/kaggle/working`, поэтому попадают во вкладку **Output** после сохранения версии.

## Что нужно перед запуском

1. **Рекомендуемый вариант для Kaggle**: прикрепить к notebook архив репозитория `top-papers-graph*.zip` **или** распакованную папку репозитория как Kaggle dataset. Тогда notebook вообще не зависит от GitHub.
2. Если локальный репозиторий не прикреплён, notebook попробует сделать `git clone` из `repo_git_url`, но только при доступном интернете и рабочем DNS.
3. Для скачивания экспертного ZIP через `gdown` нужен **Internet = ON**.
4. Заполните `CFG` в следующей ячейке.
5. Нажмите **Save Version**.

## Что происходит во время запуска

1. Notebook находит или клонирует репозиторий.
2. Устанавливает только недостающие зависимости.
3. Скачивает ZIP с экспертным YAML.
4. Извлекает YAML и запускает runner-скрипт с аргументами CLI.
5. Складывает результаты в `/kaggle/working/...`.


In [ ]:
# Конфиг для Kaggle Save Version / Run All
from pathlib import Path

CFG = {
    # --- где искать архив/папку репозитория ---
    # Если пусто, notebook сам попробует найти top-papers-graph*.zip или top-papers-graph* под /kaggle/input и /kaggle/working.
    "repo_archive_path": "",
    "repo_dir_path": "/kaggle/working/top-papers-graph",
    # Если локальный архив/папка не найдены, notebook может клонировать репозиторий отсюда.
    "repo_git_url": "https://github.com/top-papers/top-papers-graph.git",
    "repo_git_branch": "main",

    # --- скачивание ZIP с экспертным YAML через gdown ---
    # Укажите ЛИБО gdown_url, ЛИБО gdown_file_id.
    "gdown_url": "",
    "gdown_file_id": "",
    # Если YAML лежит внутри ZIP не в корне, можно уточнить шаблон.
    "yaml_glob": "*.y*ml",

    # --- базовые пути ---
    "workspace_dir": "/kaggle/working/task3_dual_gdown_workspace",
    "output_dir": "/kaggle/working/task3_dual_local_blind_ab_script",

    # --- входные данные Task 3 ---
    # Обычно query можно не задавать, если topic уже есть в YAML.
    "query": "",
    # Альтернатива или дополнение к YAML topic.
    "identifiers": [],
    "identifiers_file": "",
    # Если processed papers уже подготовлены заранее и подключены как dataset — укажите путь.
    "processed_dir": "",

    # --- параметры pipeline ---
    "domain_id": "science",
    "submission_id": "",
    "search_limit": 25,
    "top_papers": 12,
    "top_hypotheses": 8,
    "candidate_top_k": 16,
    "top_pairs": 8,
    "annoy_n_trees": 32,
    "annoy_top_k": 6,
    "include_multimodal": True,
    "run_vlm": True,
    # На Kaggle обычно безопаснее ставить heuristic, чтобы не тащить лишние зависимости для temporal GNN.
    "edge_mode": "auto",
    "link_prediction_backend": "heuristic",
    "hf_offline": False,
    "create_offline_form": True,
    "create_expert_bundle": True,

    # --- Kaggle-safe VLM defaults ---
    # Жёсткий тайм-аут ожидания ответа от local VLM worker для одной страницы.
    "local_vlm_request_timeout_seconds": 300,
    # Отдельный тайм-аут на первый preload модели в worker: холодный старт 7B/8B на Kaggle может идти заметно дольше.
    "local_vlm_startup_timeout_seconds": 900,
    # Умеренный DPI уменьшает размер PNG страниц и риск зависания/OOM на Kaggle.
    "pdf_render_dpi": 110,
    # Ограничение размера картинки, подаваемой в VLM.
    "vlm_max_pixels": 768 * 28 * 28,
    # Ограничение генерации для page-level captioning, чтобы VLM не зависал на длинном выводе.
    "vlm_max_new_tokens": 192,
    # Общий cache для HF-моделей в /kaggle/working.
    "hf_home": "/kaggle/working/.hf",

    # --- необязательные text-LLM overrides ---
    "llm_provider": None,
    "llm_model": None,
    "g4f_model": None,

    # --- Model α ---
    "model_a_owner_label": "base_local_model",
    "model_a_local_text_model": "",
    "model_a_vlm_backend": "qwen2_vl",
    "model_a_vlm_model_id": "Qwen/Qwen2.5-VL-7B-Instruct",
    "model_a_vlm_model_url": "https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct",

    # --- Model β ---
    "model_b_owner_label": "finetuned_local_model",
    "model_b_local_text_model": "",
    "model_b_vlm_backend": "qwen3_vl",
    "model_b_vlm_model_id": "Qwen/Qwen3-VL-8B-Instruct",
    "model_b_vlm_model_url": "https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct",
}

CFG


In [ ]:
import importlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

WORKSPACE_DIR = Path(CFG["workspace_dir"]).expanduser()
OUTPUT_DIR = Path(CFG["output_dir"]).expanduser()
HF_HOME = Path(str(CFG.get("hf_home") or "/kaggle/working/.hf")).expanduser()
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_HOME))
os.environ.setdefault("HF_HUB_CACHE", str(HF_HOME / "hub"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(HF_HOME / "hub"))
os.environ.setdefault("LOCAL_VLM_REQUEST_TIMEOUT_SECONDS", str(CFG.get("local_vlm_request_timeout_seconds") or 120))
os.environ.setdefault("SCIREASON_LOCAL_VLM_REQUEST_TIMEOUT_SECONDS", os.environ.get("LOCAL_VLM_REQUEST_TIMEOUT_SECONDS", "300"))
os.environ.setdefault("LOCAL_VLM_STARTUP_TIMEOUT_SECONDS", str(CFG.get("local_vlm_startup_timeout_seconds") or 900))
os.environ.setdefault("SCIREASON_LOCAL_VLM_STARTUP_TIMEOUT_SECONDS", os.environ.get("LOCAL_VLM_STARTUP_TIMEOUT_SECONDS", "900"))
os.environ.setdefault("PDF_RENDER_DPI", str(CFG.get("pdf_render_dpi") or 110))
os.environ.setdefault("VLM_MAX_PIXELS", str(CFG.get("vlm_max_pixels") or (768 * 28 * 28)))
os.environ.setdefault("VLM_MAX_NEW_TOKENS", str(CFG.get("vlm_max_new_tokens") or 192))
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



def _is_repo_root(path: Path) -> bool:
    return path.is_dir() and (path / "pyproject.toml").exists() and (path / "src" / "scireason").exists()


def _find_local_repo_source():
    explicit_archive = str(CFG.get("repo_archive_path") or "").strip()
    if explicit_archive:
        p = Path(explicit_archive)
        if p.exists():
            return ("archive", p)

    explicit_dir = str(CFG.get("repo_dir_path") or "").strip()
    if explicit_dir:
        p = Path(explicit_dir)
        if _is_repo_root(p):
            return ("dir", p)

    search_roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), Path("/mnt/data")]
    patterns = ("top-papers-graph*.zip", "top-papers-graph-main*", "top-papers-graph*")
    for base in search_roots:
        if not base.exists():
            continue
        for pattern in patterns:
            for candidate in sorted(base.rglob(pattern)):
                if candidate.is_file() and candidate.suffix == ".zip":
                    return ("archive", candidate)
                if _is_repo_root(candidate):
                    return ("dir", candidate)
    return (None, None)


def _git_host_from_url(url: str) -> str:
    text = (url or "").strip()
    if not text:
        return ""
    if "://" in text:
        from urllib.parse import urlparse
        return (urlparse(text).hostname or "").strip()
    if ":" in text and "@" in text and not text.startswith("/"):
        tail = text.split("@", 1)[-1]
        return tail.split(":", 1)[0].strip()
    return ""


def _host_resolves(host: str) -> bool:
    host = (host or "").strip()
    if not host:
        return False
    try:
        import socket
        socket.gethostbyname(host)
        return True
    except Exception:
        return False


def _clone_repo_from_git(git_url: str, branch: str, target_dir: Path) -> Path:
    git_url = (git_url or "").strip()
    branch = (branch or "main").strip() or "main"
    if not git_url:
        raise FileNotFoundError("repo_git_url пустой, клонирование невозможно.")
    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    cmd = ["git", "clone", "--depth", "1", "--branch", branch, git_url, str(target_dir)]
    print("[repo] clone from git:", " ".join(cmd))
    subprocess.check_call(cmd)
    if not _is_repo_root(target_dir):
        raise FileNotFoundError(f"После git clone не найден корень репозитория: {target_dir}")
    return target_dir


def _ensure_repo_source():
    kind, source_path = _find_local_repo_source()
    if kind and source_path:
        return kind, source_path

    git_url = str(CFG.get("repo_git_url") or "").strip()
    git_branch = str(CFG.get("repo_git_branch") or "main").strip() or "main"
    host = _git_host_from_url(git_url)
    if host and _host_resolves(host):
        clone_target = WORKSPACE_DIR / "git_clone" / "top-papers-graph"
        try:
            repo_dir = _clone_repo_from_git(git_url, git_branch, clone_target)
            return ("dir", repo_dir)
        except Exception as exc:
            raise FileNotFoundError(
                "Не удалось получить репозиторий из GitHub. "
                "Прикрепите архив/папку репозитория как Kaggle dataset или проверьте Internet/DNS. "
                f"Причина: {exc}"
            ) from exc

    raise FileNotFoundError(
        "Не удалось найти локальный архив/папку top-papers-graph, а GitHub недоступен по DNS. "
        "Чтобы избежать ошибки `Could not resolve host: github.com`, прикрепите репозиторий как Kaggle dataset "
        "или включите Internet и проверьте доступность github.com."
    )

kind, source_path = _ensure_repo_source()

repo_extract_dir = WORKSPACE_DIR / "repo"
if repo_extract_dir.exists():
    shutil.rmtree(repo_extract_dir)
repo_extract_dir.mkdir(parents=True, exist_ok=True)

if kind == "archive":
    print(f"[repo] extract archive: {source_path}")
    with zipfile.ZipFile(source_path, "r") as zf:
        zf.extractall(repo_extract_dir)
    repo_candidates = [p for p in repo_extract_dir.rglob("*") if p.is_dir() and (p / "pyproject.toml").exists() and (p / "src" / "scireason").exists()]
    if not repo_candidates:
        raise FileNotFoundError("После распаковки архива не найден корень репозитория")
    REPO_ROOT = sorted(repo_candidates, key=lambda p: len(str(p)))[0]
else:
    REPO_ROOT = source_path

SRC_DIR = REPO_ROOT / "src"
os.environ["PYTHONPATH"] = os.pathsep.join([str(SRC_DIR), str(REPO_ROOT), str(os.environ.get("PYTHONPATH") or "")]).strip(os.pathsep)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("[repo] root =", REPO_ROOT)

# Ставим только отсутствующие пакеты, не обновляя без необходимости системные пакеты Kaggle.
REQUIRED_MODULES = {
    "yaml": "pyyaml",
    "gdown": "gdown",
    "typer": "typer",
    "rich": "rich",
    "dotenv": "python-dotenv",
    "pydantic": "pydantic",
    "pydantic_settings": "pydantic-settings",
    "httpx": "httpx",
    "tenacity": "tenacity",
    "networkx": "networkx",
    "numpy": "numpy",
    "pypdf": "pypdf",
    "neo4j": "neo4j",
    "qdrant_client": "qdrant-client",
    "litellm": "litellm",
    "langchain_core": "langchain-core",
    "langgraph": "langgraph",
    "dateutil": "python-dateutil",
    "dateparser": "dateparser",
    "annoy": "annoy",
    "PIL": "pillow",
    "fitz": "pymupdf",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "sentencepiece": "sentencepiece",
    "qwen_vl_utils": "qwen-vl-utils",
    "g4f": "g4f",
    "langchain_community": "langchain-community",
    "langchain_text_splitters": "langchain-text-splitters",
    "torch_geometric": "torch-geometric",
}

missing = []
for module_name, package_name in REQUIRED_MODULES.items():
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(package_name)

# editable install без зависимостей — чтобы subprocess с `python -m scireason...` работал стабильно.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_ROOT)])

if missing:
    print("[pip] installing missing packages:", ", ".join(missing))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        *missing,
    ])
else:
    print("[pip] all required packages already available")

yaml_zip_path = WORKSPACE_DIR / "expert_yaml_bundle.zip"
gdown_url = str(CFG.get("gdown_url") or "").strip()
gdown_file_id = str(CFG.get("gdown_file_id") or "").strip()
if not gdown_url and not gdown_file_id:
    raise ValueError("Укажите gdown_url или gdown_file_id в CFG")

if yaml_zip_path.exists():
    yaml_zip_path.unlink()

print("[gdown] download ->", yaml_zip_path)



In [ ]:
# Эта ячейка специально использует !gdown, как было запрошено.
yaml_zip_target = str(yaml_zip_path)

if gdown_url:
    !gdown --fuzzy "{gdown_url}" -O "{yaml_zip_target}"
else:
    drive_url = f"https://drive.google.com/uc?id={gdown_file_id}"
    !gdown --fuzzy "{drive_url}" -O "{yaml_zip_target}"



In [ ]:
import json
import shlex
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

if not yaml_zip_path.exists():
    raise FileNotFoundError(f"ZIP с экспертным YAML не был скачан: {yaml_zip_path}")

if not zipfile.is_zipfile(yaml_zip_path):
    preview = yaml_zip_path.read_text(encoding="utf-8", errors="ignore")[:400]
    raise RuntimeError(
        "gdown скачал файл, который не является ZIP-архивом. "
        "Проверьте публичность ссылки Google Drive / квоту и содержимое ответа. "
        f"Первые символы файла: {preview!r}"
    )

yaml_extract_dir = WORKSPACE_DIR / "expert_yaml"
if yaml_extract_dir.exists():
    shutil.rmtree(yaml_extract_dir)
yaml_extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(yaml_zip_path, "r") as zf:
    zf.extractall(yaml_extract_dir)

yaml_glob = str(CFG.get("yaml_glob") or "*.y*ml").strip() or "*.y*ml"
yaml_candidates = sorted(yaml_extract_dir.rglob(yaml_glob))
yaml_candidates = [p for p in yaml_candidates if p.is_file()]
if not yaml_candidates:
    raise FileNotFoundError(f"В ZIP не найден YAML по шаблону: {yaml_glob}")

TRAJECTORY_PATH = yaml_candidates[0]
RUNNER_PATH = REPO_ROOT / "scripts" / "kaggle" / "run_task3_dual_local_blind_ab.py"
if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Не найден runner script: {RUNNER_PATH}")


def _append_arg(cmd, key, value):
    if value is None:
        return
    if isinstance(value, bool):
        cmd.append(f"--{key}" if value else f"--no-{key}")
        return
    text = str(value).strip()
    if not text:
        return
    cmd.extend([f"--{key}", text])


cmd = [sys.executable, str(RUNNER_PATH)]
_append_arg(cmd, "trajectory", TRAJECTORY_PATH)
_append_arg(cmd, "out-dir", OUTPUT_DIR)
_append_arg(cmd, "domain-id", CFG.get("domain_id"))
_append_arg(cmd, "submission-id", CFG.get("submission_id"))
_append_arg(cmd, "query", CFG.get("query"))
_append_arg(cmd, "identifiers", ",".join(CFG.get("identifiers") or []))
_append_arg(cmd, "identifiers-file", CFG.get("identifiers_file"))
_append_arg(cmd, "processed-dir", CFG.get("processed_dir"))
_append_arg(cmd, "search-limit", CFG.get("search_limit"))
_append_arg(cmd, "top-papers", CFG.get("top_papers"))
_append_arg(cmd, "top-hypotheses", CFG.get("top_hypotheses"))
_append_arg(cmd, "candidate-top-k", CFG.get("candidate_top_k"))
_append_arg(cmd, "top-pairs", CFG.get("top_pairs"))
_append_arg(cmd, "annoy-n-trees", CFG.get("annoy_n_trees"))
_append_arg(cmd, "annoy-top-k", CFG.get("annoy_top_k"))
_append_arg(cmd, "include-multimodal", bool(CFG.get("include_multimodal", True)))
_append_arg(cmd, "run-vlm", bool(CFG.get("run_vlm", True)))
_append_arg(cmd, "edge-mode", CFG.get("edge_mode"))
_append_arg(cmd, "link-prediction-backend", CFG.get("link_prediction_backend"))
_append_arg(cmd, "hf-offline", bool(CFG.get("hf_offline", False)))
_append_arg(cmd, "create-offline-form", bool(CFG.get("create_offline_form", True)))
_append_arg(cmd, "create-expert-bundle", bool(CFG.get("create_expert_bundle", True)))
_append_arg(cmd, "llm-provider", CFG.get("llm_provider"))
_append_arg(cmd, "llm-model", CFG.get("llm_model"))
_append_arg(cmd, "g4f-model", CFG.get("g4f_model"))

_append_arg(cmd, "model-a-owner-label", CFG.get("model_a_owner_label"))
_append_arg(cmd, "model-a-local-text-model", CFG.get("model_a_local_text_model"))
_append_arg(cmd, "model-a-vlm-backend", CFG.get("model_a_vlm_backend"))
_append_arg(cmd, "model-a-vlm-model-id", CFG.get("model_a_vlm_model_id"))

_append_arg(cmd, "model-b-owner-label", CFG.get("model_b_owner_label"))
_append_arg(cmd, "model-b-local-text-model", CFG.get("model_b_local_text_model"))
_append_arg(cmd, "model-b-vlm-backend", CFG.get("model_b_vlm_backend"))
_append_arg(cmd, "model-b-vlm-model-id", CFG.get("model_b_vlm_model_id"))


print("[model α] backend/id =", CFG.get("model_a_vlm_backend"), CFG.get("model_a_vlm_model_id"))
print("[local VLM] request/startup timeout =", os.environ.get("LOCAL_VLM_REQUEST_TIMEOUT_SECONDS"), os.environ.get("LOCAL_VLM_STARTUP_TIMEOUT_SECONDS"))
print("[local VLM] max pixels / max new tokens =", os.environ.get("VLM_MAX_PIXELS"), os.environ.get("VLM_MAX_NEW_TOKENS"))
print("[model α] card =", CFG.get("model_a_vlm_model_url") or "")
print("[model β] backend/id =", CFG.get("model_b_vlm_backend"), CFG.get("model_b_vlm_model_id"))
print("[model β] card =", CFG.get("model_b_vlm_model_url") or "")
print("[yaml] selected =", TRAJECTORY_PATH)
print("[runner] path =", RUNNER_PATH)
print("[command]")
print(" ".join(shlex.quote(part) for part in cmd))

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(REPO_ROOT / "src"),
    str(REPO_ROOT),
    str(env.get("PYTHONPATH") or ""),
]).strip(os.pathsep)

after_path = OUTPUT_DIR / "task3_dual_run_manifest.json"

def _ensure_root_offline_html(manifest_payload: dict, output_dir: Path, manifest_path: Path) -> str | None:
    artifacts = manifest_payload.get("artifacts") if isinstance(manifest_payload.get("artifacts"), dict) else {}
    existing = artifacts.get("offline_form_export_html")
    if existing and Path(existing).exists():
        return str(Path(existing))
    offline_form = artifacts.get("offline_form")
    if not offline_form:
        return None
    src = Path(str(offline_form))
    if not src.exists() or not src.is_file():
        return None
    dst = output_dir.parent / f"{output_dir.name}__offline_review.html"
    if src.resolve() != dst.resolve():
        shutil.copy2(src, dst)
    artifacts["offline_form_export_html"] = str(dst)
    manifest_payload["artifacts"] = artifacts
    manifest_path.write_text(json.dumps(manifest_payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return str(dst)

result = subprocess.run(cmd, cwd=str(REPO_ROOT), env=env, check=False)
if result.returncode != 0:
    error_path = OUTPUT_DIR / "task3_dual_run_error.json"
    if error_path.exists():
        print(error_path.read_text(encoding="utf-8", errors="ignore"))
    raise SystemExit(f"Runner завершился с кодом {result.returncode}")

if after_path.exists():
    manifest = json.loads(after_path.read_text(encoding="utf-8"))
    exported_html = _ensure_root_offline_html(manifest, OUTPUT_DIR, after_path)
    print("\n[done] run_manifest =", after_path)
    print(json.dumps(manifest.get("artifacts", {}), ensure_ascii=False, indent=2))
    if exported_html:
        print("[done] offline review html =", exported_html)
else:
    print("[warn] run manifest not found:", after_path)



## Где искать результаты на Kaggle

После `Save Version` откройте справа вкладку **Output**. Основные файлы будут лежать в каталоге:

`/kaggle/working/task3_dual_local_blind_ab_script`

Обычно там будут:

- `variant_alpha/`
- `variant_beta/`
- `task3_dual_run_manifest.json`
- `task3_dual_local_blind_ab_script__kaggle_outputs.zip`
- `task3_dual_local_blind_ab_script__offline_review.html`
- blind review HTML inside review bundle
- owner key
- expert ZIP
